In [1]:
# Importing necessary libraries
import os
import sys
from pathlib import Path
import argparse
import logging
import time
import json
import importlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is on the Python path so scripts/enhancement imports w

In [2]:
#Data read 

orig_path = r"C:\Users\hisham\Desktop\Nizam_Hisham\MNL\Data\processed\fr\2016\fr_2016.parquet"
df_orig = pd.read_parquet(orig_path)  # uses pyarrow/fastparquet if available

df_orig.info() # Display dataframe information
print(df_orig.head()) # Display first few rows of the dataframe

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10952 entries, 0 to 10951
Columns: 146 entries, income_market to bchcc_input
dtypes: float64(78), int16(3), int64(64), object(1)
memory usage: 12.0+ MB
   income_market  ypr_input  pdi00_input  dwt_input  afc_input  lhw_input  \
0     631.666670        0.0          0.0   8113.981    0.00000         15   
1    2989.166630        0.0          0.0   8113.981    0.00000         42   
2       0.000000        0.0          0.0   8113.981    0.00000          0   
3       0.000000        0.0          0.0   8113.981    0.00000          0   
4    4265.416667        0.0          0.0   6520.411  359.71223         50   

   kivho_input  data_year  yemxp_input  ate_input  ...       idhh  les_orig  \
0        299.0       2016      0.00000          1  ...  1483000.0       2.0   
1        299.0       2016    597.83333          1  ...  1483000.0       3.0   
2          0.0       2016      0.00000          1  ...  1483000.0       6.0   
3          0.0     

In [3]:
# Notebook cell — paste and run
from pathlib import Path
import pandas as pd, numpy as np
from IPython.display import display

base = Path(r"C:\Users\hisham\Desktop\Nizam_Hisham\MNL\Data\processed\fr\2016")
singles_p = base / "singles_RURO_ready.parquet"
couples_p = base / "couples_RURO_ready.parquet"

# Existence check
print("singles file:", singles_p.exists(), singles_p)
print("couples file:", couples_p.exists(), couples_p)

# Load
df_s = pd.read_parquet(singles_p)
df_c = pd.read_parquet(couples_p)
print("singles shape:", df_s.shape, "couples shape:", df_c.shape)

# Quick peek
display(df_s.head())
print(df_s.info(memory_usage="deep"))

# Key diagnostic variables
keys = ["loc_raw", "loc_ruro", "loc4", "is_worker", "ruro_sample", "wage_ruro", "drgn1"]
for k in keys:
    exists = k in df_s.columns
    miss = (df_s[k].isna().mean()) if exists else None
    nunq = df_s[k].nunique(dropna=True) if exists else None
    print(f"{k:12} in df: {exists:5}  missing_pct: {miss}  nunique: {nunq}")

# Value counts / distributions
if "loc4" in df_s.columns:
    print("\nloc4 value counts:")
    print(df_s["loc4"].value_counts(dropna=False).sort_index())

if "loc_ruro" in df_s.columns:
    print("\nloc_ruro value counts (sample):")
    print(df_s["loc_ruro"].value_counts(dropna=False).sort_index()[:20])

if "is_worker" in df_s.columns:
    print("\nis_worker distribution:")
    print(df_s["is_worker"].value_counts(dropna=False).sort_index())

# Wage summary for positive wages
if "wage_ruro" in df_s.columns:
    w = pd.to_numeric(df_s["wage_ruro"], errors="coerce")
    wpos = w[w > 0]
    print(f"\nwage_ruro > 0: n={len(wpos)} median={wpos.median():.3f} mean={wpos.mean():.3f} min={wpos.min():.3f} max={wpos.max():.3f}")

# Region dummies coverage
reg_cols = sorted([c for c in df_s.columns if c.startswith("reg_nuts1_")])
print("\nregion dummy cols:", reg_cols)
if reg_cols:
    print(df_s[reg_cols].sum().sort_values())

# Save a small sample CSV for quick external inspection (optional)
sample_p = base / "singles_RURO_ready_sample.csv"
df_s.sample(min(500, len(df_s)), random_state=1).to_csv(sample_p, index=False)
print("Wrote sample CSV:", sample_p)

singles file: True C:\Users\hisham\Desktop\Nizam_Hisham\MNL\Data\processed\fr\2016\singles_RURO_ready.parquet
couples file: True C:\Users\hisham\Desktop\Nizam_Hisham\MNL\Data\processed\fr\2016\couples_RURO_ready.parquet
singles shape: (2395, 581) couples shape: (8557, 581)


,idhh,idperson,idmother,idfather,idpartner,idorighh,idorigperson,dag,dgn,dec,...,reg_nuts1_1,reg_nuts1_2,reg_nuts1_3,reg_nuts1_4,reg_nuts1_5,reg_nuts1_6,reg_nuts1_7,reg_nuts1_8,reg_nuts1_9,reg_nuts1_10
0,1495800.0,149580001.0,0.0,0.0,0.0,1495800.0,149580001.0,51.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,0
1,1495800.0,149580002.0,149580001.0,0.0,0.0,1495800.0,149580002.0,18.0,1.0,4.0,...,1,0,0,0,0,0,0,0,0,0
2,1495800.0,149580003.0,149580001.0,0.0,0.0,1495800.0,149580003.0,11.0,0.0,3.0,...,1,0,0,0,0,0,0,0,0,0
3,1496401.0,149640101.0,0.0,0.0,0.0,1496401.0,149640002.0,40.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,0
4,1498400.0,149840001.0,0.0,0.0,0.0,1498400.0,149840001.0,57.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2395 entries, 0 to 2394
Columns: 581 entries, idhh to reg_nuts1_10
dtypes: bool(4), float64(465), int16(7), int64(94), int8(10), object(1)
memory usage: 10.4 MB
None
loc_raw      in df:     1  missing_pct: 0.0  nunique: 11
loc_ruro     in df:     1  missing_pct: 0.0  nunique: 11
loc4         in df:     1  missing_pct: 0.0  nunique: 6
is_worker    in df:     1  missing_pct: 0.0  nunique: 2
ruro_sample  in df:     1  missing_pct: 0.0  nunique: 2
wage_ruro    in df:     1  missing_pct: 0.3453027139874739  nunique: 1512
drgn1        in df:     1  missing_pct: 0.0  nunique: 8

loc4 value counts:
loc4
-2      7
-1    827
 1    454
 2    239
 3    148
 4    720
Name: count, dtype: int64

loc_ruro value counts (sample):
loc_ruro
-1    827
 0      7
 1     60
 2    332
 3    328
 4    148
 5    239
 6     26
 7    123
 8    136
 9    169
Name: count, dtype: int64

is_worker distribution:
is_worker
0     827
1    1568
Name: count, dtype: int64

w

In [4]:
# =============================================================================
# CELL 1: Setup and Configuration
# =============================================================================
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, HTML

# Add server directory to path
SERVER_DIR = Path(r"C:\Users\hisham\Desktop\Nizam_Hisham\MNL\server")
if str(SERVER_DIR) not in sys.path:
    sys.path.insert(0, str(SERVER_DIR))

# Configuration
BASE_DIR = Path(r"C:\Users\hisham\Desktop\Nizam_Hisham\MNL")
PROCESSED_DIR = BASE_DIR / "Data" / "processed" / "fr" / "2016"
RAW_DIR = BASE_DIR / "Data" / "raw"
EUROMOD_DIR = BASE_DIR / "EUROMOD_RELEASES_J1.0+" / "EUROMOD_RELEASES_J1.0+"
YEAR = 2016

print(f"Processed dir exists: {PROCESSED_DIR.exists()}")
print(f"Raw dir exists: {RAW_DIR.exists()}")
print(f"EUROMOD dir exists: {EUROMOD_DIR.exists()}")

# List available files
if PROCESSED_DIR.exists():
    print("\nFiles in processed dir:")
    for f in sorted(PROCESSED_DIR.glob("*.parquet")):
        print(f"  {f.name}")

Processed dir exists: True
Raw dir exists: True
EUROMOD dir exists: True

Files in processed dir:
  couples_RURO_ready.parquet
  couples_RURO_ready_RURO_draws.parquet
  fr_2016.parquet
  fr_2016_couples.parquet
  fr_2016_singles.parquet
  fr_2016_singles_female.parquet
  fr_2016_singles_male.parquet
  singles_RURO_draws_from_mod.parquet
  singles_RURO_draws_from_orig.parquet
  singles_RURO_ready.parquet
  singles_RURO_ready_RURO_draws.parquet


In [5]:
# =============================================================================
# CELL 2: Test Helper Functions
# =============================================================================
from typing import List, Dict, Any, Tuple
import traceback

class PipelineTest:
    """Helper class for pipeline testing."""
    
    def __init__(self):
        self.results: List[Dict[str, Any]] = []
    
    def test(self, name: str, condition: bool, details: str = ""):
        """Record a test result."""
        status = "✅ PASS" if condition else "❌ FAIL"
        self.results.append({
            "name": name,
            "passed": condition,
            "status": status,
            "details": details
        })
        print(f"{status}: {name}" + (f" — {details}" if details else ""))
        return condition
    
    def summary(self):
        """Print summary of all tests."""
        passed = sum(1 for r in self.results if r["passed"])
        total = len(self.results)
        print(f"\n{'='*60}")
        print(f"SUMMARY: {passed}/{total} tests passed")
        if passed < total:
            print("\nFailed tests:")
            for r in self.results:
                if not r["passed"]:
                    print(f"  - {r['name']}: {r['details']}")
        print('='*60)
        return passed == total

tester = PipelineTest()

In [6]:
# =============================================================================
# CELL 3: Test Stage 1 Output (france_data_prep.py)
# =============================================================================
print("=" * 60)
print("STAGE 1: Testing france_data_prep.py outputs")
print("=" * 60)

# Load Stage 1 outputs
stage1_singles = PROCESSED_DIR / f"fr_{YEAR}_singles.parquet"
stage1_couples = PROCESSED_DIR / f"fr_{YEAR}_couples.parquet"

# Check files exist
tester.test("Stage1: singles file exists", stage1_singles.exists(), str(stage1_singles))
tester.test("Stage1: couples file exists", stage1_couples.exists(), str(stage1_couples))

if stage1_singles.exists() and stage1_couples.exists():
    df_s1_singles = pd.read_parquet(stage1_singles)
    df_s1_couples = pd.read_parquet(stage1_couples)
    
    # Required columns for Stage 2
    STAGE1_REQUIRED = [
        "idperson", "idhh", "dag", "dgn", "les", "lhw",
        "hh_IsHead", "hh_IsPartner", "ruro_decider",
    ]
    STAGE1_WAGE_COLS = ["yivwg", "wage_final"]  # At least one required
    STAGE1_YEAR_COLS = ["data_year", "system_year", "input_year"]  # At least one
    STAGE1_EDUC_COLS = ["deh", "dehde"]  # At least one
    STAGE1_REGION_COLS = ["drgn1", "drgn2"]  # At least one for region dummies
    
    for df, name in [(df_s1_singles, "singles"), (df_s1_couples, "couples")]:
        # Core columns
        missing_core = [c for c in STAGE1_REQUIRED if c not in df.columns]
        tester.test(
            f"Stage1 {name}: core columns present",
            len(missing_core) == 0,
            f"missing: {missing_core}" if missing_core else ""
        )
        
        # Wage column (at least one)
        has_wage = any(c in df.columns for c in STAGE1_WAGE_COLS)
        tester.test(f"Stage1 {name}: wage column present", has_wage, 
                   f"need one of {STAGE1_WAGE_COLS}")
        
        # Year column (at least one)
        has_year = any(c in df.columns for c in STAGE1_YEAR_COLS)
        tester.test(f"Stage1 {name}: year column present", has_year,
                   f"need one of {STAGE1_YEAR_COLS}")
        
        # Education column
        has_educ = any(c in df.columns for c in STAGE1_EDUC_COLS)
        tester.test(f"Stage1 {name}: education column present", has_educ,
                   f"need one of {STAGE1_EDUC_COLS}")
        
        # Region column
        has_region = any(c in df.columns for c in STAGE1_REGION_COLS)
        tester.test(f"Stage1 {name}: region column present", has_region,
                   f"need one of {STAGE1_REGION_COLS}")
        
        # Occupation column
        has_loc = "loc" in df.columns
        tester.test(f"Stage1 {name}: occupation (loc) present", has_loc)
        
        # Data integrity
        tester.test(
            f"Stage1 {name}: no duplicate (idhh, idperson)",
            not df.duplicated(subset=["idhh", "idperson"]).any()
        )
        
        # Year sanity check
        for yc in STAGE1_YEAR_COLS:
            if yc in df.columns:
                y = pd.to_numeric(df[yc], errors="coerce")
                valid_years = y.between(1900, 2100).all()
                tester.test(f"Stage1 {name}: {yc} in valid range", valid_years,
                           f"min={y.min()}, max={y.max()}")
                break
        
        # ruro_decider consistency
        if "ruro_decider" in df.columns and "hh_IsHead" in df.columns:
            rd = df["ruro_decider"].fillna(0).astype(int)
            hd = df["hh_IsHead"].fillna(0).astype(int)
            pt = df.get("hh_IsPartner", pd.Series(0, index=df.index)).fillna(0).astype(int)
            expected = ((hd == 1) | (pt == 1)).astype(int)
            match = (rd == expected).all()
            tester.test(f"Stage1 {name}: ruro_decider = head OR partner", match)

    print(f"\nStage 1 singles: {len(df_s1_singles)} rows, {df_s1_singles['idhh'].nunique()} households")
    print(f"Stage 1 couples: {len(df_s1_couples)} rows, {df_s1_couples['idhh'].nunique()} households")

STAGE 1: Testing france_data_prep.py outputs
✅ PASS: Stage1: singles file exists — C:\Users\hisham\Desktop\Nizam_Hisham\MNL\Data\processed\fr\2016\fr_2016_singles.parquet
✅ PASS: Stage1: couples file exists — C:\Users\hisham\Desktop\Nizam_Hisham\MNL\Data\processed\fr\2016\fr_2016_couples.parquet
✅ PASS: Stage1 singles: core columns present
✅ PASS: Stage1 singles: wage column present — need one of ['yivwg', 'wage_final']
✅ PASS: Stage1 singles: year column present — need one of ['data_year', 'system_year', 'input_year']
✅ PASS: Stage1 singles: education column present — need one of ['deh', 'dehde']
✅ PASS: Stage1 singles: region column present — need one of ['drgn1', 'drgn2']
✅ PASS: Stage1 singles: occupation (loc) present
✅ PASS: Stage1 singles: no duplicate (idhh, idperson)
✅ PASS: Stage1 singles: data_year in valid range — min=2016, max=2016
✅ PASS: Stage1 singles: ruro_decider = head OR partner
✅ PASS: Stage1 couples: core columns present
✅ PASS: Stage1 couples: wage column present